[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab08.ipynb)

# Bab 8 — Deteksi Objek: Menghitung Sel Darah (BCCD) dengan Faster R-CNN MobileNetV3
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4**, lalu **Runtime → Run all**.

Isi:
1. **BCCD** (Shenggan, 2017; lisensi MIT): 364 citra apusan darah 640×480, anotasi VOC (RBC, WBC, Platelets), pembagian resmi `ImageSets/Main` (train/val/test).
2. Konsep: IoU, NMS, mAP@0,5 dan mAP@0,5:0,95 (COCO), dihitung dengan `torchmetrics` (*backend* pycocotools).
3. **Faster R-CNN MobileNetV3-Large FPN** dari `torchvision` (BSD-3; bobot pra-latih COCO), varian resolusi penuh vs varian **320 px** (seluler).
4. Hitung sel otomatis vs anotasi per citra (MAE hitung, Bland–Altman), latensi GPU vs CPU.

**Versi 2 (28-10-2026)** — perubahan terhadap `bab08.ipynb` (eksperimen, data, model, dan metrik sama):
- **Mode deterministik PyTorch dimatikan** untuk notebook ini. Dalam mode itu, `roi_align` torchvision di GPU beralih ke implementasi Python
  yang dikompilasi `torch.compile` (lambat, kompilasi ulang untuk bentuk tensor yang berubah) — penyebab utama run v1 sangat lama.
  Seed tetap dipasang, tetapi hasil antar-run dapat berbeda sedikit.
- 12 epoch (bukan 24), evaluasi mAP validasi tiap 3 epoch (+ epoch terakhir); citra disimpan sekali di GPU (uint8), augmentasi di GPU.
- Latensi CPU: 1 pemanasan + 5 ulangan (bukan 3 + 10).
- Batas waktu pengaman per run latih (dicatat bila tercapai), cetak progres per epoch dengan ETA, dan **waktu setiap sel** dicetak otomatis.

Jalur YOLO (Ultralytics, AGPL-3.0) **tidak** dipakai di notebook ini; lihat catatan lisensi di buku.
Hasil → `/content/hasil_bab08/` → `/content/hasil_bab08.zip`. Hanya untuk pembelajaran, **bukan** untuk penggunaan klinis.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 8.0 Lingkungan

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB08_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB08_OUT", "/content/hasil_bab08"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
# --- v2: cetak waktu SETIAP sel secara otomatis (hook IPython) ---
WAKTU_SEL = []
try:
    _ip = get_ipython()
    def _sel_mulai(info=None):
        global _T_SEL; _T_SEL = time.perf_counter()
        print(f"[sel mulai {time.strftime('%H:%M:%S')}]", flush=True)
    def _sel_selesai(hasil=None):
        dt = time.perf_counter() - _T_SEL; tot = time.perf_counter() - T_MULAI
        WAKTU_SEL.append({"sel_ke": len(WAKTU_SEL) + 1, "detik": round(dt, 2), "kumulatif_detik": round(tot, 1)})
        print(f"[sel selesai] {dt:.1f} s | kumulatif notebook {tot/60:.1f} menit", flush=True)
    for _ev, _fn in (("pre_run_cell", _sel_mulai), ("post_run_cell", _sel_selesai)):
        for _f in list(_ip.events.callbacks[_ev]):
            if getattr(_f, "__name__", "") == _fn.__name__: _ip.events.unregister(_ev, _f)
        _ip.events.register(_ev, _fn)
    _T_SEL = time.perf_counter()
except NameError:
    pass

with catat_waktu("1_pip_install"):
    import importlib.util
    perlu = [p for p, m in (("torchmetrics>=1.4", "torchmetrics"), ("pycocotools", "pycocotools")) if importlib.util.find_spec(m) is None]
    if perlu:
        print("pip install:", perlu, flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *perlu], check=True, timeout=900)
import numpy as np, pandas as pd, torch, torchvision, sklearn, matplotlib, matplotlib.ticker, matplotlib.patches as mpatches
import matplotlib.pyplot as plt, xml.etree.ElementTree as ET, torchmetrics
from PIL import Image
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection import MeanAveragePrecision
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__, "torchmetrics": torchmetrics.__version__,
         "numpy": np.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))

SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()

# --- v2: matikan mode deterministik (lihat catatan di atas). Seed tetap dipasang. ---
def seed_semua(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
seed_semua()
print("algoritme deterministik:", torch.are_deterministic_algorithms_enabled(), "| cudnn.deterministic:", torch.backends.cudnn.deterministic)

## 8.1 Data BCCD (format Pascal VOC)
Setiap berkas XML memuat `<object>` dengan nama kelas dan `<bndbox>` (xmin, ymin, xmax, ymax). Kotak yang rusak (lebar/tinggi ≤ 1 piksel)
dibuang dan dicatat jumlahnya. Indeks kelas: 0 = latar (khas torchvision), 1 = RBC, 2 = WBC, 3 = Platelets.

In [ ]:
KELAS = ["latar", "RBC", "WBC", "Platelets"]
with catat_waktu("2_muat_data"):
    pastikan_dataset("bccd", ["BCCD_Dataset-master.zip"])
    akar = ekstrak_lokal("bccd", "BCCD_Dataset-master.zip") / "BCCD_Dataset-master" / "BCCD"
    def baca_voc(nama):
        r = ET.parse(akar / "Annotations" / f"{nama}.xml").getroot(); kotak, label, rusak = [], [], 0
        for o in r.iter("object"):
            b = o.find("bndbox"); v = [float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
            v = [min(max(v[0], 0), 639), min(max(v[1], 0), 479), min(max(v[2], 0), 640), min(max(v[3], 0), 480)]
            if v[2] - v[0] <= 1 or v[3] - v[1] <= 1: rusak += 1; continue
            kotak.append(v); label.append(KELAS.index(o.find("name").text.strip()))
        return np.array(kotak, np.float32).reshape(-1, 4), np.array(label, np.int64), rusak
    DATA, RUSAK = {}, 0
    for s in ("train", "val", "test"):
        nama = (akar / "ImageSets" / "Main" / f"{s}.txt").read_text().split()
        nama = [n for n in nama if (akar / "JPEGImages" / f"{n}.jpg").exists()]
        if SMOKE: nama = nama[:{"train": 16, "val": 6, "test": 6}[s]]
        item = []
        for n in nama:
            k, l, rs = baca_voc(n); RUSAK += rs
            img = np.array(Image.open(akar / "JPEGImages" / f"{n}.jpg").convert("RGB"))
            item.append({"nama": n, "img": img, "x": torch.as_tensor(img).permute(2, 0, 1).contiguous().to(DEV),   # v2: uint8 di GPU, sekali
                         "boxes": k, "labels": l, "b_t": torch.as_tensor(k).to(DEV), "l_t": torch.as_tensor(l).to(DEV)})
        DATA[s] = item; print(f"{s}: {len(item)} citra dimuat", flush=True)
T81 = pd.DataFrame([{"split": s, "n_citra": len(v), **{f"n_{KELAS[c]}": int(sum((it["labels"] == c).sum() for it in v)) for c in (1, 2, 3)},
                     "rerata_RBC_per_citra": float(np.mean([(it["labels"] == 1).sum() for it in v]))} for s, v in DATA.items()])
T81.to_csv(OUT / "tabel_8_1_data.csv", index=False); print("kotak rusak dibuang:", RUSAK)
WARNA = {1: "red", 2: "blue", 3: "lime"}
def gambar_kotak(a, img, boxes, labels, skor=None, judul=""):
    a.imshow(img); a.axis("off"); a.set_title(judul, fontsize=8)
    for i, (b, l) in enumerate(zip(boxes, labels)):
        a.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=WARNA[int(l)], lw=1))
        if skor is not None: a.text(b[0], b[1], f"{skor[i]:.2f}", color=WARNA[int(l)], fontsize=5)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, it in zip(ax, DATA["train"][:3]): gambar_kotak(a, it["img"], it["boxes"], it["labels"], judul=f"{it['nama']}: {len(it['labels'])} kotak")
fig.legend(handles=[mpatches.Patch(color=WARNA[c], label=KELAS[c]) for c in (1, 2, 3)], loc="lower center", ncol=3)
plt.tight_layout(); plt.savefig(OUT / "gambar_8_1_contoh_anotasi.png", bbox_inches="tight"); plt.show()
uk = np.concatenate([np.c_[it["boxes"][:, 2] - it["boxes"][:, 0], it["boxes"][:, 3] - it["boxes"][:, 1], it["labels"]] for it in DATA["train"]])
fig, a = plt.subplots(figsize=(5, 3.5))
for c in (1, 2, 3): a.scatter(uk[uk[:, 2] == c, 0], uk[uk[:, 2] == c, 1], s=4, alpha=0.5, c=WARNA[c], label=KELAS[c])
a.set_xlabel("lebar kotak (px)"); a.set_ylabel("tinggi kotak (px)"); a.legend(); plt.tight_layout(); plt.savefig(OUT / "gambar_8_2_ukuran_kotak.png", bbox_inches="tight"); plt.show(); T81

## 8.2 IoU dan NMS dalam beberapa baris

In [ ]:
a = torch.tensor([[10., 10, 60, 60]]); b = torch.tensor([[30., 30, 80, 80], [10, 10, 60, 60], [100, 100, 120, 120]])
print("IoU:", torchvision.ops.box_iou(a, b).numpy().round(3))
kotak = torch.tensor([[10., 10, 60, 60], [12, 12, 62, 62], [100, 100, 150, 150]]); skor = torch.tensor([0.9, 0.8, 0.7])
print("NMS (IoU 0,5) menyisakan indeks:", torchvision.ops.nms(kotak, skor, 0.5).tolist())

## 8.3 Pelatihan Faster R-CNN MobileNetV3-Large FPN
- Kepala klasifikasi COCO (91 kelas) diganti `FastRCNNPredictor(…, 4)`.
- SGD (lr 0,01, momentum 0,9, wd 1e-4), *warm-up* 1 epoch + kosinus, batch 4, AMP; augmentasi flip horizontal/vertikal (kotak ikut dibalik) dan kecerahan ±15 %.
- Model terbaik dipilih dari **mAP@0,5 validasi** (dievaluasi tiap 2 epoch).
- Dua varian: **resolusi penuh** (`min_size=800`, bawaan) dan **320 px** (`fasterrcnn_mobilenet_v3_large_320_fpn`).

In [ ]:
def buat_model(varian):
    fn = fasterrcnn_mobilenet_v3_large_fpn if varian == "penuh" else fasterrcnn_mobilenet_v3_large_320_fpn
    try: m = fn(weights="DEFAULT")
    except Exception as e:
        if not SMOKE: raise
        print("SMOKE: bobot COCO gagal diunduh, pakai acak:", e); m = fn(weights=None, weights_backbone=None)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, len(KELAS))
    m.roi_heads.detections_per_img = 100            # sama dengan maxDets COCO; satu citra BCCD memuat hingga puluhan RBC
    if SMOKE: m.transform.min_size, m.transform.max_size = (256,), 341
    return m.to(DEV)
def ke_input(it, aug=False, g=None):
    x = it["x"].float().div(255); b = it["b_t"].clone(); H, W = x.shape[1:]
    if aug:
        if float(torch.rand(1, generator=g)) < 0.5: x = x.flip(2); b[:, [0, 2]] = W - b[:, [2, 0]]
        if float(torch.rand(1, generator=g)) < 0.5: x = x.flip(1); b[:, [1, 3]] = H - b[:, [3, 1]]
        x = (x * (1 + (float(torch.rand(1, generator=g)) * 2 - 1) * 0.15)).clamp(0, 1)   # v2: skalar Python (generator di CPU, x di GPU)
    return x, {"boxes": b, "labels": it["l_t"]}
@torch.no_grad()
def prediksi(model, item, bs=8):
    model.eval(); out = []
    for i in range(0, len(item), bs):
        xs = [ke_input(it)[0] for it in item[i:i + bs]]
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"): o = model(xs)
        out += [{k: v.float().cpu() if v.is_floating_point() else v.cpu() for k, v in d.items()} for d in o]
    return out
def hitung_map(pred, item, kelas=True):
    mt = MeanAveragePrecision(iou_type="bbox", class_metrics=kelas, backend="pycocotools")   # maxDets bawaan [1, 10, 100]
    mt.update(pred, [{"boxes": torch.as_tensor(it["boxes"]), "labels": torch.as_tensor(it["labels"])} for it in item])
    r = mt.compute(); hasil = {"mAP50": float(r["map_50"]), "mAP50_95": float(r["map"])}
    if kelas:
        for c, ap in zip(r["classes"].tolist(), r["map_per_class"].tolist()): hasil[f"AP50_95_{KELAS[c]}"] = ap
    return hasil
EPOCH, EVAL_TIAP = 12, 3                                # v2: 24 -> 12 epoch, evaluasi tiap 3 epoch
BATAS_DETIK = {"penuh": 300, "320": 150}               # v2: pengaman per run (normalnya tidak tercapai)
def latih_det(varian, seed, n_ep=EPOCH, bs=4, lr=0.01):
    seed_semua(seed); m = buat_model(varian); g = torch.Generator().manual_seed(seed)
    if SMOKE: n_ep = 2
    par = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.SGD(par, lr=lr, momentum=0.9, weight_decay=1e-4)
    n_it = int(np.ceil(len(DATA["train"]) / bs)); tot = n_ep * n_it
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: min(1.0, (i + 1) / n_it) * 0.5 * (1 + np.cos(np.pi * i / tot)))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda"); log = []; best = (-1, None, 0); T0 = time.perf_counter(); henti = False
    for ep in range(1, n_ep + 1):
        m.train(); t0 = time.perf_counter(); perm = torch.randperm(len(DATA["train"]), generator=g).tolist(); tl = 0
        for i in range(0, len(perm), bs):
            xs, ts = zip(*[ke_input(DATA["train"][j], aug=True, g=g) for j in perm[i:i + bs]])
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                ld = m(list(xs), list(ts)); loss = sum(ld.values())
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()
        if DEV.type == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0; baris = {"model": f"{varian} | seed {seed}", "epoch": ep, "loss_latih": tl / n_it, "detik_epoch": dt}
        if time.perf_counter() - T0 > BATAS_DETIK[varian] and ep < n_ep and not SMOKE:
            henti = True; print(f"!! batas waktu {BATAS_DETIK[varian]} s tercapai pada epoch {ep}; pelatihan dihentikan", flush=True)
        if ep % EVAL_TIAP == 0 or ep == n_ep or henti:
            mv = hitung_map(prediksi(m, DATA["val"]), DATA["val"], kelas=False); baris.update({"mAP50_val": mv["mAP50"], "mAP50_95_val": mv["mAP50_95"]})
            if mv["mAP50"] > best[0]: best = (mv["mAP50"], {k: v.detach().clone() for k, v in m.state_dict().items()}, ep)
        baris["berhenti_batas_waktu"] = henti; log.append(baris)
        el = time.perf_counter() - T0
        print(f"[{varian} s{seed}] epoch {ep}/{n_ep} loss {baris['loss_latih']:.3f} | {dt:.1f} s/epoch | "
              + (f"mAP50_val {baris['mAP50_val']:.3f} | " if "mAP50_val" in baris else "")
              + f"berjalan {el/60:.1f} mnt, ETA run {el/ep*(n_ep-ep)/60:.1f} mnt", flush=True)
        if henti: break
    m.load_state_dict(best[1]); return m, pd.DataFrame(log), best[2]
VARIAN = {"penuh": [0] if SMOKE else [0, 1], "320": [0]}
HASIL, LOG, MODEL = [], {}, {}
for v, seeds in VARIAN.items():
    for s in seeds:
        print(f"=== latih varian {v}, seed {s} ===", flush=True)
        with catat_waktu(f"latih_{v}_s{s}"): m, lg, ep = latih_det(v, s)
        with catat_waktu(f"evaluasi_{v}_s{s}"): pu = prediksi(m, DATA["test"]); mu = hitung_map(pu, DATA["test"])
        LOG[(v, s)] = lg; MODEL[(v, s)] = (m, pu)
        HASIL.append({"varian": v, "seed": s, "epoch_terbaik": ep, "epoch_dijalankan": int(lg.epoch.max()), "berhenti_batas_waktu": bool(lg.berhenti_batas_waktu.any()), "mAP50_val_terbaik": float(lg.mAP50_val.max()), "detik_per_epoch": float(lg.detik_epoch.mean()),
                      "n_param_juta": sum(p.numel() for p in m.parameters()) / 1e6, **{f"{k}_uji": val for k, val in mu.items()}})
        print(HASIL[-1])
T82 = pd.DataFrame(HASIL); T82.to_csv(OUT / "tabel_8_2_map.csv", index=False); pd.concat(LOG.values()).to_csv(OUT / "log_epoch.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for (v, s), lg in LOG.items():
    ax[0].plot(lg.epoch, lg.loss_latih, label=f"{v} s{s}"); d = lg.dropna(subset=["mAP50_val"]); ax[1].plot(d.epoch, d.mAP50_val, "o-", ms=3, label=f"{v} s{s}")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss latih (jumlah 4 komponen)"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("mAP@0,5 val")
for a in ax: a.legend(fontsize=7); a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(OUT / "gambar_8_3_kurva_latih.png", bbox_inches="tight"); plt.show(); T82

## 8.4 Visualisasi prediksi dan AP per kelas
Model utama: varian **penuh** dengan mAP@0,5 validasi tertinggi. Kotak prediksi ditampilkan untuk skor ≥ 0,5.

In [ ]:
PIL = max([k for k in MODEL if k[0] == "penuh"], key=lambda k: T82[(T82.varian == k[0]) & (T82.seed == k[1])].mAP50_val_terbaik.iloc[0])
model, PU = MODEL[PIL]; torch.save(model.state_dict(), OUT / "fasterrcnn_mnv3_bccd.pt")
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for j, it in enumerate(DATA["test"][:3]):
    gambar_kotak(ax[0, j], it["img"], it["boxes"], it["labels"], judul=f"anotasi: {len(it['labels'])} kotak")
    p = PU[j]; k = p["scores"] >= 0.5
    gambar_kotak(ax[1, j], it["img"], p["boxes"][k].numpy(), p["labels"][k].numpy(), p["scores"][k].numpy(), judul=f"prediksi (skor ≥ 0,5): {int(k.sum())} kotak")
fig.legend(handles=[mpatches.Patch(color=WARNA[c], label=KELAS[c]) for c in (1, 2, 3)], loc="lower center", ncol=3)
plt.tight_layout(); plt.savefig(OUT / "gambar_8_4_prediksi_vs_anotasi.png", bbox_inches="tight"); plt.show()
r = T82[(T82.varian == PIL[0]) & (T82.seed == PIL[1])].iloc[0]
T83 = pd.DataFrame([{"kelas": KELAS[c], "AP50_95_uji": r.get(f"AP50_95_{KELAS[c]}_uji", np.nan)} for c in (1, 2, 3)])
# AP@0,5 per kelas: hitung ulang per kelas
for c in (1, 2, 3):
    pc = [{"boxes": p["boxes"][p["labels"] == c], "scores": p["scores"][p["labels"] == c], "labels": p["labels"][p["labels"] == c]} for p in PU]
    ic = [{"boxes": it["boxes"][it["labels"] == c], "labels": it["labels"][it["labels"] == c]} for it in DATA["test"]]
    T83.loc[T83.kelas == KELAS[c], "AP50_uji"] = hitung_map(pc, ic, kelas=False)["mAP50"]
T83.to_csv(OUT / "tabel_8_3_ap_per_kelas.csv", index=False); T83

## 8.5 Hitung sel otomatis vs anotasi
Ambang skor per kelas dipilih pada set **validasi** untuk meminimalkan MAE hitung, lalu diterapkan pada set uji.
Catatan: anotasi RBC di BCCD diketahui tidak selalu lengkap (sel bertumpuk atau di tepi), sehingga "acuan" hitung juga mengandung galat.

In [ ]:
with catat_waktu("3_prediksi_val_hitung"): PV = prediksi(model, DATA["val"])
AMB = {}
for c in (1, 2, 3):
    yv = np.array([(it["labels"] == c).sum() for it in DATA["val"]])
    kand = np.round(np.arange(0.2, 0.95, 0.05), 2)
    mae = [np.abs(np.array([int(((p["labels"] == c) & (p["scores"] >= t)).sum()) for p in PV]) - yv).mean() for t in kand]
    AMB[c] = float(kand[int(np.argmin(mae))])
baris = []; fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, c in zip(ax, (1, 2, 3)):
    y = np.array([(it["labels"] == c).sum() for it in DATA["test"]]); yh = np.array([int(((p["labels"] == c) & (p["scores"] >= AMB[c])).sum()) for p in PU])
    d = yh - y; baris.append({"kelas": KELAS[c], "ambang_skor_val": AMB[c], "rerata_acuan": y.mean(), "rerata_prediksi": yh.mean(), "MAE_hitung": np.abs(d).mean(),
                              "bias": d.mean(), "LoA_bawah": d.mean() - 1.96 * d.std(ddof=1), "LoA_atas": d.mean() + 1.96 * d.std(ddof=1),
                              "persen_citra_tepat": float((d == 0).mean() * 100)})
    a.scatter(y + np.random.default_rng(0).uniform(-0.2, 0.2, len(y)), yh, s=10, alpha=0.6); lim = [0, max(y.max(), yh.max()) + 1]; a.plot(lim, lim, "k--", lw=1)
    a.set_xlabel(f"hitung anotasi {KELAS[c]}"); a.set_ylabel("hitung prediksi"); a.set_title(f"{KELAS[c]} (ambang {AMB[c]}): MAE {np.abs(d).mean():.2f}")
plt.tight_layout(); plt.savefig(OUT / "gambar_8_5_hitung_sel.png", bbox_inches="tight"); plt.show()
T84 = pd.DataFrame(baris); T84.to_csv(OUT / "tabel_8_4_hitung_sel.csv", index=False); T84

## 8.6 Latensi inferensi (satu citra 640×480): GPU vs CPU

In [ ]:
def latensi(m, dev, n=20, panas=3):
    m = m.to(dev).eval(); x = [DATA["test"][0]["x"].float().div(255).to(dev)]
    with torch.no_grad():
        for _ in range(panas): m(x)
        if dev.type == "cuda": torch.cuda.synchronize()
        t = []
        for _ in range(n):
            t0 = time.perf_counter(); m(x)
            if dev.type == "cuda": torch.cuda.synchronize()
            t.append(time.perf_counter() - t0)
    return float(np.median(t) * 1000)
baris = []
with catat_waktu("4_latensi"):
    for (v, s), (m, _) in MODEL.items():
        if s != 0: continue
        r = {"varian": v, "ms_GPU": latensi(m, DEV) if DEV.type == "cuda" else np.nan}
        n_thr = torch.get_num_threads(); torch.set_num_threads(2)
        r["ms_CPU_2thread"] = latensi(m, torch.device("cpu"), n=2 if SMOKE else 5, panas=1); torch.set_num_threads(n_thr)   # v2: 1 + 5 ulangan
        baris.append(r); m.to(DEV); print(r, flush=True)
T85 = pd.DataFrame(baris); T85.to_csv(OUT / "tabel_8_5_latensi.csv", index=False); T85

## 8.7 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(data=T81.to_dict("records"), kotak_rusak=RUSAK, map=T82.to_dict("records"), model_terpilih=list(PIL), ap_per_kelas=T83.to_dict("records"), hitung_sel=T84.to_dict("records"), latensi=T85.to_dict("records"), versi_notebook="v2", waktu_per_sel=WAKTU_SEL, epoch=EPOCH, eval_tiap=EVAL_TIAP, deterministik=False))))
(OUT / "ringkasan_bab08.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass